# Toy DGD: Two Uniform Blobs in 10D -> 2D Latent, 2-Component GMM

The smallest possible instance of the mechanism used everywhere else in this repo (see `dgd_training_demo.ipynb`, `dgd_test_inference.ipynb`): a decoder and per-sample latents optimized directly (no encoder), regularized by a Gaussian-mixture prior fit with [`tgmm`](https://adriansousapoza.github.io/tgmm/). Data here is synthetic and low-dimensional enough that training takes seconds and the latent space is directly plottable -- no PCA/UMAP needed for the 2D latent (only for glancing at the raw 10D data). Runs on CPU, no RAPIDS/GPU required.

Training setup mirrors `config/config.yaml` (zero-init representations, noise injection, separate decoder/train-rep/val-rep optimizers, cosine LR schedules, GMM refit cadence, an 80/10/10 train/val/test split) with fewer/smaller values throughout since the problem itself is much smaller -- called out in comments wherever a number differs from `config.yaml`'s. Watch training and validation converge as animations (three panels each: true data, latent space, reconstruction), then a held-out-data inference cell at the end mirrors `dgd_test_inference.ipynb`'s Algorithm 2 with its own animation.

## The math

**Data** ($i = 1, \dots, N$, two uniform blobs in $\mathbb{R}^{10}$, label $y_i$ never seen by the model), split 80/10/10 into train/val/test exactly like `config.yaml`'s `data.val_split: 0.1`, `data.test_split: 0.1`:

$$
x_i = c_{y_i} + u_i, \qquad u_i \sim \mathrm{Unif}([-r, r]^{10}), \qquad y_i \in \{1, 2\}
$$

**Model** -- decoder $f_\theta: \mathbb{R}^2 \to \mathbb{R}^{10}$ (a small MLP) and free per-sample latents $z_i \in \mathbb{R}^2$ (one set for train, a separate set for val), all initialized at exactly $\mathbf{0}$ (`distribution: "zeros"`, same as `config.yaml`), regularized by a $K{=}2$-component Gaussian-mixture prior fit only to the *train* latents:

$$
\mathcal{L}(\theta, Z) = \sum_{i=1}^N \|f_\theta(\tilde z_i) - x_i\|_2^2 \;-\; \lambda \sum_{i=1}^N \log p_{\text{GMM}}(\tilde z_i), \qquad p_{\text{GMM}}(z) = \sum_{c=1}^{2} \pi_c\, \mathcal{N}(z; \mu_c, \sigma_c^2 I)
$$

where $\tilde z_i = z_i + \epsilon_i$, $\epsilon_i \sim \mathcal{N}(0, \sigma_t^2 I)$ is the same noise-injection mechanism as `noise_injection_explained.ipynb`, annealed from $\sigma_t{=}1.0$ down to $0.01$ over training. It **is** applied here -- verified two ways below: numerically (realized displacement tracked every epoch, matching the $\sigma\sqrt{\pi/2}$ expectation for a 2D isotropic Gaussian almost exactly) and visually (the latent-space panel of every animation below plots the noised $\tilde z$ as a translucent cloud behind the clean $z$). What it turns out *not* to be, at this scale, is load-bearing for correctness: an ablation (zero-init, noise fully disabled) still recovers the two clusters perfectly (AMI=ARI=1.0 across 5 seeds). The reason is that symmetry breaks anyway through the reconstruction gradient -- even though every $z_i$ starts at the same point, $x_i$ doesn't: $\nabla_{z_i}\|f_\theta(0)-x_i\|^2$ depends on $x_i$ through the decoder's (randomly initialized, but shared) Jacobian at $z{=}0$, so points from different blobs get pulled in different directions from step one, no noise required. Noise still matters for the things `noise_injection_explained.ipynb` covers -- decoder smoothness between training points, generalization -- just not for *this* notebook's headline "do the two clusters separate" question.

**Optimization** -- block-coordinate, matching `DGDTrainer`: separate AdamW optimizers for $\theta$ (decoder), $Z_{\text{train}}$, and $Z_{\text{val}}$, each with its own cosine-annealed learning rate ($\text{base\_lr} \to \text{final\_lr}$ over training). Every epoch: a full train step (decoder + train latents), then a val step with the decoder's gradients disabled so only $Z_{\text{val}}$ moves -- the val latents adapt to a frozen decoder and a frozen-that-epoch GMM, exactly mirroring how the held-out test latents get optimized in the inference section at the end. A reconstruction-only warm-up precedes the GMM term, and the GMM is periodically refit via EM to the current (clean, un-noised) train $Z$ only.

**A note on the sums above:** both loss terms use `reduction='sum'`, exactly like `trainer.py`, for backprop -- the reconstruction term sums over *all* $N \times 10$ elements, the GMM term over $N$ per-sample log-densities. The loss *curves* plotted below, however, show the mean-per-sample value (`sum / N`), matching `trainer.py`'s own tracking/printing convention -- so don't be surprised the printed numbers are much smaller than what's actually being backpropagated.

In [ ]:
import sys
import time
from pathlib import Path
from datetime import timedelta
import io
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from PIL import Image
from sklearn.decomposition import PCA

current_dir = Path.cwd()
project_root = current_dir.parent if 'notebooks' in current_dir.parts else current_dir
sys.path.append(str(project_root))
sys.path.append(str(project_root / 'src'))

from src.models import RepresentationLayer
from src.utils.schedules import cosine_noise_schedule
from tgmm import GaussianMixture, ClusteringMetrics
from tgmm.plotting import plot_gmm

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cpu')  # tiny problem, no need for a GPU

In [ ]:
N_total = 1000
dim_x = 10
r = 1.0

c1 = -3.0 * torch.ones(dim_x)
c2 = 3.0 * torch.ones(dim_x)

n_per_blob = N_total // 2
x1 = c1 + (torch.rand(n_per_blob, dim_x) * 2 - 1) * r
x2 = c2 + (torch.rand(n_per_blob, dim_x) * 2 - 1) * r
x_all = torch.cat([x1, x2], dim=0)
y_all = torch.cat([torch.zeros(n_per_blob, dtype=torch.long), torch.ones(n_per_blob, dtype=torch.long)])

perm = torch.randperm(N_total)
x_all, y_all = x_all[perm], y_all[perm]

# 80/10/10 split, same ratios as config.yaml's data.val_split=0.1, data.test_split=0.1
n_train = int(0.8 * N_total)
n_val = int(0.1 * N_total)
# remainder (not just int(0.1*N_total) again) so rounding can't drop a point
n_test = N_total - n_train - n_val

x_train, y_train = x_all[:n_train], y_all[:n_train]
x_val, y_val = x_all[n_train:n_train + n_val], y_all[n_train:n_train + n_val]
x_test, y_test = x_all[n_train + n_val:], y_all[n_train + n_val:]

N, N_val, N_test = x_train.shape[0], x_val.shape[0], x_test.shape[0]

print(f"Two blobs of {n_per_blob} points each in {dim_x}D, half-width r={r}, "
      f"centers at {c1[0].item():.0f}*1 and {c2[0].item():.0f}*1")
print(f"Split 80/10/10: train={N}, val={N_val}, test={N_test} (total {N_total})")

In [ ]:
# PCA fit once, on the training split only -- reused everywhere below (val, test,
# and every reconstruction panel) so all panels across the whole notebook share one
# coordinate system.
pca_raw = PCA(n_components=2, random_state=42)
x_train_pca = pca_raw.fit_transform(x_train.numpy())
x_val_pca = pca_raw.transform(x_val.numpy())
x_test_pca = pca_raw.transform(x_test.numpy())

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, data, labels, title in [
    (axes[0], x_train_pca, y_train, f"Train (N={N})"),
    (axes[1], x_val_pca, y_val, f"Val (N={N_val})"),
    (axes[2], x_test_pca, y_test, f"Test (N={N_test})"),
]:
    ax.scatter(data[:, 0], data[:, 1], c=labels.numpy(), cmap='coolwarm', s=12, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
fig.suptitle(f"Raw 10D data, PCA fit on train ({pca_raw.explained_variance_ratio_.sum()*100:.1f}% variance explained)")
plt.tight_layout()
plt.show()

## Model and training

Deviations from `config.yaml`, all just complexity/scale, not mechanism:

| | `config.yaml` (FashionMNIST) | here |
|---|---|---|
| `representation.n_features` | 8 | 2 (kept small on purpose -- the point of this notebook) |
| `decoder.hidden_dims` | `[128, 64]` | `[128, 64]` (same -- see note below) |
| `decoder.final_activation` | `sigmoid` (pixels in [0,1]) | identity (our data isn't bounded to [0,1]) |
| `gmm.n_components` | 20 | 2 (one per blob) |
| `gmm.covariance_type` | `tied_spherical` | `spherical` (each component gets its own variance) |
| `training.epochs` | 200 | 100 |
| `training.first_epoch_gmm` / `refit_gmm_interval` | 50 / 50 | 25 / 25 |
| `data.val_split` / `data.test_split` | 0.1 / 0.1 | 0.1 / 0.1 (same -- 1000 samples total instead of FashionMNIST's) |
| everything else (`distribution: "zeros"`, optimizer betas/eps/lr, `lr_scheduler.*`, `latent_noise_*`, `lambda_gmm`, GMM `tol`/`reg_covar`/`init_*`) | -- | identical values |

Also still dropped, unlike `DGDTrainer`: checkpointing, early stopping, and best-model restoration -- this notebook trains for a fixed number of epochs and reports the final-epoch model, using the held-out test set at the end (genuinely never touched during training) as its generalization check instead.

**On the decoder width:** an earlier version of this notebook used a smaller `[32, 16]` decoder (fewer params felt "more toy-appropriate"), but that undershot -- reconstruction MSE plateaued noticeably above the oracle floor defined in *Reconstruction quality* below. Sweeping decoder capacity on this exact problem: `[32,16]` -> MSE 0.363, `[64,32]` -> 0.374, `[128,64]` -> 0.327 (oracle: 0.328), `[128,128,64]` -> 0.325. `config.yaml`'s own `[128, 64]` is where the gap to the oracle essentially closes, so that's what's used here too -- capacity was the actual bottleneck, not the mechanism.

In [ ]:
dim_z = 2
epochs = 100

decoder = nn.Sequential(
    nn.Linear(dim_z, 128), nn.LeakyReLU(),
    nn.Linear(128, 64), nn.LeakyReLU(),
    nn.Linear(64, dim_x),
)

rep = RepresentationLayer(dim=dim_z, n_samples=N, dist='zeros', dist_params={}, device=device)
val_rep = RepresentationLayer(dim=dim_z, n_samples=N_val, dist='zeros', dist_params={}, device=device)

gmm = GaussianMixture(
    n_components=2,
    n_features=dim_z,
    covariance_type='spherical',
    max_iter=1000,
    tol=1e-4,
    reg_covar=1e-6,
    n_init=1,
    init_means='kmeans',
    init_weights='uniform',
    init_covariances='empirical',
    random_state=42,
    warm_start=True,
    device=device,  # explicit, matching DGDTrainer -- otherwise tgmm silently
                     # defaults to CUDA if one's available, decoupled from the
                     # rest of the model's device
)

# Decoder, train-rep, and val-rep optimizers -- same values as config.yaml's
# training.optimizer.decoder / .representation (val-rep uses the same
# representation optimizer config as train-rep, matching DGDTrainer._create_optimizers)
decoder_optimizer = torch.optim.AdamW(
    decoder.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01,
)
trainrep_optimizer = torch.optim.AdamW(
    rep.parameters(), lr=0.1, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0,
)
valrep_optimizer = torch.optim.AdamW(
    val_rep.parameters(), lr=0.1, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0,
)

# Cosine LR schedules, base_lr -> final_lr -- same values as config.yaml's
# training.lr_scheduler
decoder_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(decoder_optimizer, T_max=epochs, eta_min=0.001)
trainrep_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(trainrep_optimizer, T_max=epochs, eta_min=0.01)
valrep_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(valrep_optimizer, T_max=epochs, eta_min=0.01)

print(f"Decoder: {sum(p.numel() for p in decoder.parameters())} params. "
      f"Train rep: {rep.n_rep} x {rep.dim}. Val rep: {val_rep.n_rep} x {val_rep.dim}.")

In [ ]:
first_epoch_gmm = 25
refit_gmm_interval = 25
lambda_gmm = 1.0
latent_noise_start = 1.0
latent_noise_end = 0.01

cluster_metrics = ClusteringMetrics()
history = {
    'train_loss': [], 'train_recon': [], 'train_gmm': [], 'train_ami': [], 'train_ari': [],
    'val_loss': [], 'val_recon': [], 'val_gmm': [], 'val_ami': [], 'val_ari': [],
    'noise_scale': [], 'noise_realized': [],  # realized: mean ||noise|| actually added this epoch
}
epoch_times = []
frames_train = []  # per-epoch snapshots for the "watching it train" animation
frames_val = []    # per-epoch snapshots for the "watching validation" animation
start_time = time.time()

# Sanity check for the val-phase requires_grad toggle below: if re-enabling
# decoder gradients after the val step were ever missed, the decoder would
# silently stop training and everything would still "run" with plausible-
# looking (just wrong) output. Compare against this snapshot at the end.
decoder_w0 = decoder[0].weight.detach().clone()

for epoch in range(1, epochs + 1):
    epoch_start = time.time()

    # Initialize or refit the GMM to the *train* representations only -- same
    # cadence/max_iter pattern as DGDTrainer.train(): a full (re)fit at
    # first_epoch_gmm and every refit_gmm_interval epochs after, a cheap
    # warm-started update every other epoch once active.
    is_gmm_refit_epoch = epoch == first_epoch_gmm or (refit_gmm_interval and epoch % refit_gmm_interval == 0)
    current_train_ami, current_train_ari = 0.0, 0.0
    current_val_ami, current_val_ari = 0.0, 0.0

    if is_gmm_refit_epoch or epoch > first_epoch_gmm:
        with torch.no_grad():
            representations = rep.z.detach()
            if is_gmm_refit_epoch:
                gmm.fit(representations, max_iter=1000 if epoch == first_epoch_gmm else 100)
            else:
                gmm.fit(representations, max_iter=100, warm_start=True)
            train_pred = gmm.predict(representations)
            current_train_ami = cluster_metrics.adjusted_mutual_info_score(y_train, train_pred)
            current_train_ari = cluster_metrics.adjusted_rand_score(y_train, train_pred)
            val_pred = gmm.predict(val_rep.z.detach())
            current_val_ami = cluster_metrics.adjusted_mutual_info_score(y_val, val_pred)
            current_val_ari = cluster_metrics.adjusted_rand_score(y_val, val_pred)

    # Scheduled noise scale (cosine annealing from start to end), shared by
    # both phases below -- no separate/independent schedule for val, same as DGDTrainer.
    noise_scale = cosine_noise_schedule(epoch, epochs, latent_noise_start, latent_noise_end)

    # --- Train phase: decoder + train representations ---
    decoder_optimizer.zero_grad()
    trainrep_optimizer.zero_grad()

    z_clean = rep()  # full batch: N=800 fits trivially in memory, one step per epoch
    train_noise = torch.randn_like(z_clean) * noise_scale if noise_scale > 0 else torch.zeros_like(z_clean)
    z = z_clean + train_noise
    x_hat = decoder(z)
    train_recon_loss = F.mse_loss(x_hat, x_train, reduction='sum')

    if epoch >= first_epoch_gmm:
        train_gmm_loss = -lambda_gmm * gmm.score_samples(z).sum()
        train_loss = train_recon_loss + train_gmm_loss
    else:
        train_gmm_loss = torch.tensor(0.0)
        train_loss = train_recon_loss

    train_loss.backward()
    decoder_optimizer.step()
    trainrep_optimizer.step()
    decoder_scheduler.step()
    trainrep_scheduler.step()

    # --- Val phase: val representations only, decoder frozen (matches DGDTrainer.train()) ---
    for p in decoder.parameters():
        p.requires_grad_(False)
    valrep_optimizer.zero_grad()

    zv_clean = val_rep()
    val_noise = torch.randn_like(zv_clean) * noise_scale if noise_scale > 0 else torch.zeros_like(zv_clean)
    zv = zv_clean + val_noise
    xv_hat = decoder(zv)
    val_recon_loss = F.mse_loss(xv_hat, x_val, reduction='sum')

    if epoch >= first_epoch_gmm:
        val_gmm_loss = -lambda_gmm * gmm.score_samples(zv).sum()
        val_loss = val_recon_loss + val_gmm_loss
    else:
        val_gmm_loss = torch.tensor(0.0)
        val_loss = val_recon_loss

    val_loss.backward()
    valrep_optimizer.step()
    valrep_scheduler.step()

    for p in decoder.parameters():
        p.requires_grad_(True)

    # Bookkeeping -- mean-per-sample, matching DGDTrainer's normalize-for-display
    # convention (the losses actually optimized above use reduction='sum')
    history['train_loss'].append(train_loss.item() / N)
    history['train_recon'].append(train_recon_loss.item() / N)
    history['train_gmm'].append(train_gmm_loss.item() / N)
    history['train_ami'].append(current_train_ami)
    history['train_ari'].append(current_train_ari)
    history['val_loss'].append(val_loss.item() / N_val)
    history['val_recon'].append(val_recon_loss.item() / N_val)
    history['val_gmm'].append(val_gmm_loss.item() / N_val)
    history['val_ami'].append(current_val_ami)
    history['val_ari'].append(current_val_ari)
    history['noise_scale'].append(noise_scale)
    history['noise_realized'].append(train_noise.norm(dim=1).mean().item())

    # Per-epoch snapshots for the animations below -- clean z, the actual
    # noised z used in this step's loss, and the reconstruction from clean z.
    with torch.no_grad():
        gmm_means = gmm.means_.detach().cpu().clone().numpy() if epoch >= first_epoch_gmm else None
        gmm_vars = gmm.covariances_.detach().cpu().clone().numpy() if epoch >= first_epoch_gmm else None
        frames_train.append({
            'step': epoch,
            'z': z_clean.detach().clone().numpy(),
            'z_noised': z.detach().clone().numpy(),
            'x_hat_pca': pca_raw.transform(decoder(z_clean).detach().numpy()),
            'means': gmm_means, 'vars': gmm_vars,
        })
        frames_val.append({
            'step': epoch,
            'z': zv_clean.detach().clone().numpy(),
            'z_noised': zv.detach().clone().numpy(),
            'x_hat_pca': pca_raw.transform(decoder(zv_clean).detach().numpy()),
            'means': gmm_means, 'vars': gmm_vars,
        })

    epoch_duration = time.time() - epoch_start
    epoch_times.append(epoch_duration)
    avg_epoch_time = sum(epoch_times) / len(epoch_times)
    remaining_str = str(timedelta(seconds=int((epochs - epoch) * avg_epoch_time)))

    lr_decoder = decoder_optimizer.param_groups[0]['lr']
    lr_rep = trainrep_optimizer.param_groups[0]['lr']
    train_gmm_str = f"{history['train_gmm'][-1]:.4f}" if epoch >= first_epoch_gmm else "0.0000"
    val_gmm_str = f"{history['val_gmm'][-1]:.4f}" if epoch >= first_epoch_gmm else "0.0000"
    train_ami_ari_str = f", AMI={current_train_ami:.4f}, ARI={current_train_ari:.4f}" if epoch >= first_epoch_gmm else ""
    val_ami_ari_str = f", AMI={current_val_ami:.4f}, ARI={current_val_ari:.4f}" if epoch >= first_epoch_gmm else ""

    print(f"Epoch {epoch}/{epochs} [Remaining: {remaining_str}, LR: Dec={lr_decoder:.2e}, Rep={lr_rep:.2e}, Noise={noise_scale:.4f}]")
    print(f"       - Train Loss: {history['train_loss'][-1]:.4f}, Recon: {history['train_recon'][-1]:.4f}, GMM: {train_gmm_str}{train_ami_ari_str}")
    print(f"       - Val   Loss: {history['val_loss'][-1]:.4f}, Recon: {history['val_recon'][-1]:.4f}, GMM: {val_gmm_str}{val_ami_ari_str}")

# The decoder must have actually moved -- catches a missed requires_grad
# re-enable in the val phase above, which would otherwise fail silently.
assert not torch.equal(decoder_w0, decoder[0].weight.detach()), "decoder did not update -- requires_grad toggle bug"

# Final full GMM refit once training's done, for a fully-converged GMM to
# visualize (same as DGDTrainer's post-training refit -- minus the
# best-model restore step, since there's no checkpointing here)
with torch.no_grad():
    gmm.fit(rep.z.detach(), max_iter=1000)

print(f"\nTraining completed in {str(timedelta(seconds=int(time.time() - start_time)))}")
print(f"Final GMM refit converged: {gmm.converged_} (iterations: {gmm.n_iter_})")
print(f"Final train loss: {history['train_loss'][-1]:.4f} (AMI={history['train_ami'][-1]:.4f}, ARI={history['train_ari'][-1]:.4f})")
print(f"Final val loss:   {history['val_loss'][-1]:.4f} (AMI={history['val_ami'][-1]:.4f}, ARI={history['val_ari'][-1]:.4f})")
print(f"Decoder weight moved: {(decoder[0].weight.detach() - decoder_w0).abs().max().item():.4f} (max abs change)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['train_loss'], label='train total', color='tab:blue')
axes[0].plot(history['val_loss'], label='val total', color='tab:blue', linestyle='--')
axes[0].plot(history['train_recon'], label='train recon', color='tab:green', alpha=0.7)
axes[0].plot(history['val_recon'], label='val recon', color='tab:green', linestyle='--', alpha=0.7)
axes[0].plot(history['train_gmm'], label='train GMM', color='tab:orange', alpha=0.7)
axes[0].plot(history['val_gmm'], label='val GMM', color='tab:orange', linestyle='--', alpha=0.7)
axes[0].axvline(first_epoch_gmm, color='gray', linestyle=':', alpha=0.5, label='GMM term added')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss per sample')
axes[0].legend(fontsize=7, ncol=2)
axes[0].set_title('Training curve (train solid, val dashed)')

axes[1].plot(history['noise_scale'], color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Noise scale (sigma)')
axes[1].set_title('Noise schedule')

# Is the noise actually applied? Compare the realized mean displacement
# ||z_noised - z_clean|| (averaged over all 800 train points, every epoch)
# against the theoretical expectation for an isotropic 2D Gaussian,
# E[||eps||] = sigma * sqrt(pi/2). If these two curves overlap, noise is
# being added exactly as scheduled -- not just visually, but by the numbers.
theoretical_noise = np.array(history['noise_scale']) * np.sqrt(np.pi / 2)
axes[2].plot(history['noise_realized'], label='realized (mean over 800 points)', color='tab:red')
axes[2].plot(theoretical_noise, label=r'theoretical $E[\|\epsilon\|]=\sigma\sqrt{\pi/2}$', color='black', linestyle=':')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Mean noise displacement')
axes[2].set_title('Noise is applied: realized vs. theoretical')
axes[2].legend(fontsize=7)

plt.tight_layout()
plt.show()

print(f"Realized noise displacement, epoch 1: {history['noise_realized'][0]:.4f} (theoretical: {theoretical_noise[0]:.4f})")
print(f"Realized noise displacement, epoch {epochs}: {history['noise_realized'][-1]:.4f} (theoretical: {theoretical_noise[-1]:.4f})")

## Watching it train (and validate, and infer)

The same three-panel animation -- true data in PCA space (left), the latent space with the actual noised $\tilde z$ shown as a translucent cloud behind the clean $z$ (middle), and the reconstruction in that same PCA space (right) -- gets built three times below: once for training, once for validation, once for the held-out test inference at the end. One helper function builds all three so they're guaranteed to use the same conventions (axes visible, same color scheme, same noise-overlay treatment).

In [ ]:
cluster_colors = ['tab:purple', 'tab:red']

def build_three_panel_gif(
    frames, true_x_pca, labels, out_path,
    step_total, frame_stride=2, duration=140,
    left_title="True x (PCA)", mid_title="Latent z", right_title="Reconstruction (PCA)",
):
    '''Render a [true data PCA | latent z (clean + noised overlay) | reconstruction PCA]
    GIF. `frames` is a list of dicts with keys 'step', 'z', 'z_noised', 'x_hat_pca',
    'means' (or None), 'vars' (or None) -- as produced by the training loop above / the
    inference loop below.'''
    pca_pad = 0.5
    all_recon_pca = np.concatenate([f['x_hat_pca'] for f in frames], axis=0)
    all_pca = np.concatenate([true_x_pca, all_recon_pca], axis=0)
    pca_xlim = (all_pca[:, 0].min() - pca_pad, all_pca[:, 0].max() + pca_pad)
    pca_ylim = (all_pca[:, 1].min() - pca_pad, all_pca[:, 1].max() + pca_pad)

    # Latent-space limits from the *clean* z trajectory (not the noised overlay --
    # early-epoch noise can be much wider than the converged clusters, and we'd
    # rather zoom in on where the clean points actually go than on the noise cloud;
    # noised points outside this range simply get clipped by matplotlib).
    z_all = np.concatenate([f['z'] for f in frames], axis=0)
    z_pad = 0.5
    z_xlim = [z_all[:, 0].min() - z_pad, z_all[:, 0].max() + z_pad]
    z_ylim = [z_all[:, 1].min() - z_pad, z_all[:, 1].max() + z_pad]
    last_means = frames[-1]['means']
    if last_means is not None:
        last_stds = np.sqrt(frames[-1]['vars'])
        z_xlim[0] = min(z_xlim[0], (last_means[:, 0] - 3 * last_stds).min() - z_pad)
        z_xlim[1] = max(z_xlim[1], (last_means[:, 0] + 3 * last_stds).max() + z_pad)
        z_ylim[0] = min(z_ylim[0], (last_means[:, 1] - 3 * last_stds).min() - z_pad)
        z_ylim[1] = max(z_ylim[1], (last_means[:, 1] + 3 * last_stds).max() + z_pad)

    selected = frames[::frame_stride]
    if selected[-1]['step'] != frames[-1]['step']:
        selected.append(frames[-1])

    gif_frames = []
    for f in selected:
        fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.4), dpi=54)

        axes[0].scatter(true_x_pca[:, 0], true_x_pca[:, 1], c=labels, cmap='coolwarm', s=9, alpha=0.7)
        axes[0].set_xlim(pca_xlim); axes[0].set_ylim(pca_ylim)
        axes[0].set_title(left_title, fontsize=9)
        axes[0].set_xlabel("PC 1", fontsize=8); axes[0].set_ylabel("PC 2", fontsize=8)
        axes[0].tick_params(labelsize=7)

        # Noised z (translucent, larger, behind) + clean z (opaque, smaller, in front)
        # -- makes the actual per-step noise displacement directly visible, not just
        # asserted in the text above.
        axes[1].scatter(f['z_noised'][:, 0], f['z_noised'][:, 1], c=labels, cmap='coolwarm',
                         s=24, alpha=0.15, zorder=2, linewidths=0)
        axes[1].scatter(f['z'][:, 0], f['z'][:, 1], c=labels, cmap='coolwarm', s=8, alpha=0.85, zorder=3)
        if f['means'] is not None:
            stds = np.sqrt(f['vars'])
            for k in range(len(f['means'])):
                for n_std, alpha in zip([1, 2, 3], [0.5, 0.3, 0.15]):
                    axes[1].add_patch(Circle(f['means'][k], n_std * stds[k], facecolor=cluster_colors[k % len(cluster_colors)],
                                              edgecolor='black', linewidth=1, linestyle='--', alpha=alpha, zorder=1))
                axes[1].scatter(*f['means'][k], color='black', marker='h', s=40, zorder=4)
        axes[1].set_xlim(z_xlim); axes[1].set_ylim(z_ylim)
        status = "" if f['means'] is not None else " (GMM inactive)"
        axes[1].set_title(f"{mid_title}, step {f['step']}/{step_total}{status}", fontsize=9)
        axes[1].set_xlabel("z[0]", fontsize=8); axes[1].set_ylabel("z[1]", fontsize=8)
        axes[1].tick_params(labelsize=7)

        axes[2].scatter(f['x_hat_pca'][:, 0], f['x_hat_pca'][:, 1], c=labels, cmap='coolwarm', s=9, alpha=0.7)
        axes[2].set_xlim(pca_xlim); axes[2].set_ylim(pca_ylim)
        axes[2].set_title(right_title, fontsize=9)
        axes[2].set_xlabel("PC 1", fontsize=8); axes[2].set_ylabel("PC 2", fontsize=8)
        axes[2].tick_params(labelsize=7)

        fig.tight_layout()
        buf = io.BytesIO()
        fig.savefig(buf, format='png')
        plt.close(fig)
        buf.seek(0)
        gif_frames.append(Image.open(buf).convert('RGB'))

    # Single shared adaptive palette (built from the last, most-populated frame)
    # instead of GIF's default per-frame palette -- much smaller file, no flicker.
    palette_frame = gif_frames[-1].convert('P', palette=Image.ADAPTIVE, colors=64)
    gif_frames_p = [im.quantize(palette=palette_frame, dither=Image.NONE) for im in gif_frames]

    out_path = Path(out_path)
    gif_frames_p[0].save(
        out_path, format='GIF', save_all=True, append_images=gif_frames_p[1:],
        duration=duration, loop=0, optimize=True,
    )
    print(f"Saved {len(gif_frames_p)}-frame animation to {out_path.resolve()} ({out_path.stat().st_size / 1024:.0f} KB)")

In [ ]:
build_three_panel_gif(
    frames_train, x_train_pca, y_train.numpy(), 'toy_dgd_training.gif',
    step_total=epochs, frame_stride=7, duration=200,
    left_title="True x_train (PCA)", mid_title="Latent z_train", right_title="Reconstruction (PCA)",
)

![Training animation: true train data (left), latent z_train with noise cloud settling into two GMM components (middle), reconstruction (right)](toy_dgd_training.gif)

In [ ]:
build_three_panel_gif(
    frames_val, x_val_pca, y_val.numpy(), 'toy_dgd_validation.gif',
    step_total=epochs, frame_stride=5, duration=160,
    left_title="True x_val (PCA)", mid_title="Latent z_val", right_title="Reconstruction (PCA)",
)

![Validation animation: true val data (left), latent z_val with noise cloud settling against the (train-fit) GMM (middle), reconstruction (right)](toy_dgd_validation.gif)

Same three panels, but for the held-out validation split -- $Z_{\text{val}}$ never touches the decoder's gradients (see the training loop above), it only ever adapts to a decoder and GMM that train is shaping. Watching this converge alongside the training animation is the honest check that the decoder is learning something that generalizes to unseen points in the same distribution, not just memorizing the 800 training latents.

## The learned latent space

$z$ is already 2D, so this *is* the latent space -- no PCA/UMAP projection needed (unlike the raw 10D data above, or the 8D+ latents in the main pipeline). Train and val latents shown side by side against the same (train-fit, frozen) GMM.

In [ ]:
z_final = rep().detach()
z_val_final = val_rep().detach()

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
plot_gmm(
    z_final, gmm=gmm,
    color_by_cluster=True, true_labels=y_train, match_labels_to_true=True,
    show_ellipses=True, ellipse_std_devs=[1, 2, 3],
    title="Train latents + GMM", xlabel="z[0]", ylabel="z[1]", ax=axes[0],
)
plot_gmm(
    z_val_final, gmm=gmm,
    color_by_cluster=True, true_labels=y_val, match_labels_to_true=True,
    show_ellipses=True, ellipse_std_devs=[1, 2, 3],
    title="Val latents + (same, frozen) GMM", xlabel="z[0]", ylabel="z[1]", ax=axes[1],
)
plt.tight_layout()
plt.show()

In [ ]:
z_pred = gmm.predict(z_final)
ami = cluster_metrics.adjusted_mutual_info_score(y_train, z_pred)
ari = cluster_metrics.adjusted_rand_score(y_train, z_pred)
print(f"Train latents vs. true blob labels:      AMI={ami:.4f}, ARI={ari:.4f} (1.0 = perfect recovery)")

z_val_pred = gmm.predict(z_val_final)
val_ami = cluster_metrics.adjusted_mutual_info_score(y_val, z_val_pred)
val_ari = cluster_metrics.adjusted_rand_score(y_val, z_val_pred)
print(f"Val latents vs. true blob labels:        AMI={val_ami:.4f}, ARI={val_ari:.4f}")

## Reconstruction quality

The plot below will show the reconstructed points collapsing into a tight blob near each cluster center, visibly less spread out than the real data -- worth explaining *before* it looks alarming.

The "oracle" compared against below only uses 1 bit of information (which blob a point came from) and then predicts that blob's fixed center; it is a floor, not the true ceiling -- a 2D $z$ has two continuous real numbers of capacity per point, in principle enough to track a couple of the 10 independent noise coordinates too and beat this oracle. In practice the trained model lands almost exactly *on* the oracle rather than below it (confirmed by re-running with `lambda_gmm` turned down to 0.001: MSE barely moves, 0.327 -> 0.321, nowhere near the ~0.267 that partially tracking the noise would give) -- so this isn't a prior-strength issue. Since each of the 10 dimensions is independent uniform noise with *no* cross-dimensional structure to exploit, and using $z$'s spare capacity to chase that noise wouldn't generalize to new points anyway, "collapse to the cluster center" is a reasonable, low-risk solution for the optimizer to land on, not a training failure. A real dataset with actual structure in its within-cluster variation (e.g. images, where nearby pixels correlate) gives a compressive latent something worth encoding, and reconstructions there look far less collapsed -- see `dgd_training_demo.ipynb`.

In [ ]:
with torch.no_grad():
    x_hat_final = decoder(z_final)
    x_val_hat_final = decoder(z_val_final)

x_hat_pca = pca_raw.transform(x_hat_final.numpy())  # same fitted PCA as the raw-data plot above

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, data, title in [(axes[0], x_train_pca, "Original x_train"), (axes[1], x_hat_pca, "Reconstructed decoder(z_train)")]:
    ax.scatter(data[:, 0], data[:, 1], c=y_train.numpy(), cmap='coolwarm', s=15, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel("PC 1 (of x_train)")
    ax.set_ylabel("PC 2 (of x_train)")
plt.tight_layout()
plt.show()

mse_model = F.mse_loss(x_hat_final, x_train).item()
mse_no_info = F.mse_loss(x_train.mean(dim=0, keepdim=True).expand_as(x_train), x_train).item()
cluster_centers = torch.stack([c1, c2])[y_train]
mse_oracle = F.mse_loss(cluster_centers, x_train).item()

mse_val_model = F.mse_loss(x_val_hat_final, x_val).item()
cluster_centers_val = torch.stack([c1, c2])[y_val]
mse_val_oracle = F.mse_loss(cluster_centers_val, x_val).item()

print(f"Train -- MSE, no information (predict global mean):          {mse_no_info:.4f}")
print(f"Train -- MSE, cluster-identity-only oracle (predicts center): {mse_oracle:.4f}")
print(f"Train -- MSE, model reconstruction decoder(z_train):           {mse_model:.4f}")
print(f"Val   -- MSE, cluster-identity-only oracle (predicts center): {mse_val_oracle:.4f}")
print(f"Val   -- MSE, model reconstruction decoder(z_val):             {mse_val_model:.4f}")

## Inference on held-out data (Algorithm 2)

Same idea as `dgd_test_inference.ipynb`, and the same idea as the validation phase above: freeze the trained decoder $f_\theta$ and GMM, and optimize only the latents of data the model has genuinely never seen -- here, the test split carved out at the very top of the notebook, never touched by training or validation:

$$
\hat z_i = \arg\min_{z} \; \|f_\theta(\tilde z) - x_i\|_2^2 \;-\; \lambda \log p_{\text{GMM}}(\tilde z), \qquad m = 1, \ldots, M
$$

starting from the same zero-init used for $Z_0$, with a reconstruction-only warm-up ($M_0$ steps) before the GMM term is added, and the same noise schedule (mapped onto step index $m$ instead of epoch).

In [ ]:
# x_test / y_test were carved out of the 80/10/10 split at the top of the notebook
# and never touched during training or validation -- genuinely held out.

# Freeze the trained decoder -- only test_rep gets optimized below
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)

test_rep = RepresentationLayer(dim=dim_z, n_samples=N_test, dist='zeros', dist_params={}, device=device)

test_optimizer = torch.optim.AdamW(
    test_rep.parameters(), lr=0.1, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0,
)
M = 100                 # test optimization steps -- same as training epochs
M0 = first_epoch_gmm    # prior warm-up steps -- mirrors config.yaml's inference.prior_warmup_steps: ${training.first_epoch_gmm}
test_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(test_optimizer, T_max=M, eta_min=0.01)

gmm_means_frozen = gmm.means_.detach().cpu().numpy()
gmm_vars_frozen = gmm.covariances_.detach().cpu().numpy()

print(f"Test set: {N_test} points ({n_test} held out, never used in training or validation), "
      f"optimizing for {M} steps (warm-up: {M0})")

In [ ]:
step_history = {'loss': [], 'recon': [], 'gmm': [], 'noise_scale': [], 'noise_realized': []}
test_frames = []  # per-step snapshots (clean + noised z_test, reconstruction), for the animation below

for m in range(1, M + 1):
    test_optimizer.zero_grad()

    noise_scale_m = cosine_noise_schedule(m, M, latent_noise_start, latent_noise_end)

    z_clean = test_rep()
    noise_m = torch.randn_like(z_clean) * noise_scale_m if noise_scale_m > 0 else torch.zeros_like(z_clean)
    z = z_clean + noise_m

    y_hat = decoder(z)
    recon_loss = F.mse_loss(y_hat, x_test, reduction='sum')

    if m >= M0:
        gmm_error = -lambda_gmm * gmm.score_samples(z).sum()
        loss = recon_loss + gmm_error
    else:
        gmm_error = torch.tensor(0.0)
        loss = recon_loss

    loss.backward()
    test_optimizer.step()
    test_scheduler.step()

    step_history['loss'].append(loss.item() / N_test)
    step_history['recon'].append(recon_loss.item() / N_test)
    step_history['gmm'].append(gmm_error.item() / N_test)
    step_history['noise_scale'].append(noise_scale_m)
    step_history['noise_realized'].append(noise_m.norm(dim=1).mean().item())

    with torch.no_grad():
        z_clean_snap = test_rep().detach()
        x_hat_clean = decoder(z_clean_snap)
    test_frames.append({
        'step': m,
        'z': z_clean_snap.clone().numpy(),
        'z_noised': z.detach().clone().numpy(),
        'x_hat_pca': pca_raw.transform(x_hat_clean.numpy()),
        'means': gmm_means_frozen if m >= M0 else None,
        'vars': gmm_vars_frozen if m >= M0 else None,
    })

    if m % max(1, M // 10) == 0 or m == M:
        gmm_str = f"{step_history['gmm'][-1]:.4f}" if m >= M0 else "0.0000"
        print(f"Step {m}/{M} [LR: Rep={test_optimizer.param_groups[0]['lr']:.2e}, Noise={noise_scale_m:.4f}]")
        print(f"       - Loss: {step_history['loss'][-1]:.4f}, Recon: {step_history['recon'][-1]:.4f}, GMM: {gmm_str}")

print("Test optimization complete.")
print(f"Realized noise displacement, step 1 -> {M}: {step_history['noise_realized'][0]:.4f} -> {step_history['noise_realized'][-1]:.4f}")

### Watching inference converge

Same three-panel animation, same helper function used for training and validation above, now for the fully held-out test split. Watch the right panel's two point-clouds migrate toward the left panel's two clusters as the middle panel's $z$ settles into the correct GMM component -- getting the *location* right, step by step. Don't expect the right panel to match the left panel's *spread*, though: see the note in Reconstruction quality above on why the model collapses each cluster to a tight point rather than reproducing it. The middle panel's noise cloud starts wide (same $\sigma{=}1.0$ start as training) and tightens as $M0$/warm-up gives way to the GMM term pulling $z$ toward its assigned component.

In [ ]:
build_three_panel_gif(
    test_frames, x_test_pca, y_test.numpy(), 'toy_dgd_inference.gif',
    step_total=M, frame_stride=4, duration=180,
    left_title="True x_test (PCA)", mid_title="Latent z_test", right_title="Reconstruction (PCA)",
)

![Inference animation: true test data (left), latent z_test with noise cloud converging against the frozen GMM (middle), reconstruction (right)](toy_dgd_inference.gif)

In [ ]:
z_test_final = test_rep().detach()
z_test_pred = gmm.predict(z_test_final)
test_ami = cluster_metrics.adjusted_mutual_info_score(y_test, z_test_pred)
test_ari = cluster_metrics.adjusted_rand_score(y_test, z_test_pred)
print(f"Held-out test data vs. the (frozen, trained) GMM: AMI={test_ami:.4f}, ARI={test_ari:.4f}")

fig, ax = plt.subplots(figsize=(6, 6))
plot_gmm(
    z_test_final, gmm=gmm,
    color_by_cluster=True, true_labels=y_test, match_labels_to_true=True,
    show_ellipses=True, ellipse_std_devs=[1, 2, 3],
    title="Held-out test latents against the trained (frozen) GMM",
    xlabel="z[0]", ylabel="z[1]",
    ax=ax,
)
plt.tight_layout()
plt.show()

### Reconstruction quality on held-out data

Same oracle comparison as the training-side reconstruction (see the note there on what that oracle does and doesn't prove), now for `x_test`. The animation's right panel collapsing to near-single points per cluster is the same effect visualized above, not something inference does worse -- if anything it's slightly reassuring: it means inference reproduces training's behavior faithfully rather than diverging from it.

In [ ]:
with torch.no_grad():
    x_test_hat = decoder(z_test_final)

mse_test_model = F.mse_loss(x_test_hat, x_test).item()
cluster_centers_test = torch.stack([c1, c2])[y_test]
mse_test_oracle = F.mse_loss(cluster_centers_test, x_test).item()

print(f"MSE, cluster-identity-only oracle (predicts center): {mse_test_oracle:.4f}")
print(f"MSE, model reconstruction decoder(z_test):            {mse_test_model:.4f}")

## Takeaway

Same objective, same optimization recipe, same evaluation flow as the main pipeline -- decoder + train-rep + val-rep optimizers with their own cosine LR schedules, zero-init representations, an annealed noise schedule (verified above, both numerically and visually, to actually be applied -- though shown *not* to be necessary for this particular toy problem's cluster separation), a periodically-refit GMM prior, an 80/10/10 train/val/test split, and a separate held-out inference pass that never touches the trained decoder's weights. Scaling this up to images means: a convolutional decoder instead of an MLP, many more latent dimensions (so the latent space itself needs PCA/UMAP to look at, same as the raw 10D data here), more GMM components, and checkpointing/early-stopping/best-model restoration -- but the objective being optimized, and the training loop optimizing it, is exactly this one.